<a href="https://colab.research.google.com/github/llelus/DSA-Project/blob/main/03_ml_shap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

if not os.path.exists("/content/DSA-Project"):
    !git clone https://github.com/llelus/DSA-Project.git /content/DSA-Project
    !pip install lightgbm shap -q

os.chdir("/content/DSA-Project")
print("Working directory:", os.getcwd())
print("Data file exists:", os.path.exists("data/processed/merged_dataset.csv"))


In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
)

df = pd.read_csv("data/processed/merged_dataset.csv")
df["timestamp"]    = pd.to_datetime(df["timestamp"],    utc=True)
df["market_start"] = pd.to_datetime(df["market_start"], utc=True)
df["market_end"]   = pd.to_datetime(df["market_end"],   utc=True)

print(f"Rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")


In [ ]:
# Extract threshold price from question text
df["threshold"] = df["question"].str.extract(r"above ([\d,]+)").replace(",", "", regex=True).astype(float)

# BTC distance from threshold (%)
df["btc_to_threshold_pct"] = (df["btc_price"] - df["threshold"]) / df["threshold"] * 100

# Minute-level returns
df = df.sort_values("timestamp")
df["btc_return_1m"]  = df.groupby("conditionId")["btc_price"].pct_change(fill_method=None) * 100
df["poly_change_1m"] = df.groupby("conditionId")["yes_price"].diff()

# 15-minute rolling volatility
df["rolling_volatility_15m"] = (
    df.groupby("conditionId")["btc_price"]
    .transform(lambda x: x.pct_change(fill_method=None).rolling(window=15, min_periods=3).std() * 100)
)

# Lag features
df["poly_lag_1"] = df.groupby("conditionId")["yes_price"].shift(1)
df["poly_lag_3"] = df.groupby("conditionId")["yes_price"].shift(3)
df["btc_lag_1"]  = df.groupby("conditionId")["btc_price"].shift(1)

# Target: will Polymarket yes_price increase next minute?
df["target"] = (df.groupby("conditionId")["yes_price"].shift(-1) > df["yes_price"]).astype(int)

df = df.dropna(subset=["target", "btc_to_threshold_pct", "rolling_volatility_15m"])

print(f"Rows ready for model: {len(df)}")
print(f"Target distribution:\n{df['target'].value_counts()}")


In [ ]:
features = [
    "btc_to_threshold_pct", "btc_return_1m", "rolling_volatility_15m",
    "poly_lag_1", "poly_lag_3", "btc_lag_1", "yes_price"
]

df_model = df[features + ["target"]].dropna().reset_index(drop=True)

X = df_model[features]
y = df_model["target"]

# TimeSeriesSplit prevents data leakage
tscv = TimeSeriesSplit(n_splits=5)

all_preds = []
all_true  = []
all_probs = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = lgb.LGBMClassifier(
        n_estimators=200,
        learning_rate=0.05,
        class_weight="balanced",
        random_state=42,
        verbose=-1
    )
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]

    all_preds.extend(preds)
    all_true.extend(y_test)
    all_probs.extend(probs)

    fold_auc = roc_auc_score(y_test, probs)
    print(f"Fold {fold+1}: test={len(y_test)}, positives={y_test.sum()}, AUC={fold_auc:.3f}")

print("\n" + "=" * 50)
print(classification_report(all_true, all_preds, target_names=["No Change", "Up"]))

overall_auc   = roc_auc_score(all_true, all_probs)
avg_precision = average_precision_score(all_true, all_probs)
print(f"AUC-ROC : {overall_auc:.4f}  (0.50 = random, 1.00 = perfect)")
print(f"PR-AUC  : {avg_precision:.4f}")

# Confusion matrix
cm   = confusion_matrix(all_true, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Change", "Up"])
disp.plot(cmap="Blues")
plt.title("LightGBM — Confusion Matrix (TimeSeriesSplit)")
plt.tight_layout()
plt.savefig("data/processed/plot_confusion_matrix.png", dpi=150)
plt.show()


In [ ]:
fpr, tpr, _         = roc_curve(all_true, all_probs)
pr_prec, pr_rec, _  = precision_recall_curve(all_true, all_probs)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC Curve
axes[0].plot(fpr, tpr, color="steelblue", lw=2, label=f"AUC = {overall_auc:.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="gray", label="Random (AUC=0.50)")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve — LightGBM")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Precision-Recall Curve
baseline = sum(all_true) / len(all_true)
axes[1].plot(pr_rec, pr_prec, color="darkorange", lw=2, label=f"PR-AUC = {avg_precision:.3f}")
axes[1].axhline(y=baseline, color="gray", linestyle="--", label=f"Baseline ({baseline:.2f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve — LightGBM")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("data/processed/plot_roc_pr.png", dpi=150)
plt.show()


In [ ]:
explainer   = shap.TreeExplainer(model)
shap_values = explainer(X_test)

print("SHAP values shape:", shap_values.shape)
print("SHAP values type :", type(shap_values))


In [ ]:
# Beeswarm plot
shap.plots.beeswarm(shap_values, max_display=7, show=False)
plt.title("SHAP — Feature Importance (Up class)")
plt.tight_layout()
plt.savefig("data/processed/plot_shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()

# Bar plot
shap.plots.bar(shap_values, max_display=7, show=False)
plt.title("SHAP — Mean Feature Importance")
plt.tight_layout()
plt.savefig("data/processed/plot_shap_bar.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
import subprocess, os
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")
os.chdir("/content/DSA-Project")

subprocess.run(["git", "config", "user.email", "kadirnsy@gmail.com"])
subprocess.run(["git", "config", "user.name",  "llelus"])
subprocess.run(["git", "pull", "--rebase", "origin", "main"])
subprocess.run(["git", "add", "data/processed/"])
subprocess.run(["git", "commit", "-m", "add: ML outputs — confusion matrix, ROC/PR curves, SHAP plots"])
subprocess.run(["git", "push", f"https://{token}@github.com/llelus/DSA-Project.git", "main"])
print("Done.")
